# Notebook 2 - Regression Advanced Extension

**Expected duration:** 45-75 minutes

## Objective
Extend the guided regression workflow with model stability checks, tuning, and interpretability.

## Inputs
- Same bike-demand dataset as Notebook 1
- Prepared feature pipeline from the core workflow

## Outputs
- Cross-validation stability view
- Tuned model comparison and permutation-importance interpretation

## Checkpoint expectations
- Run CV and interpret fold variance
- Tune non-linear model and compare lift
- Complete error-slice analysis by hour


## Dataset handling

This notebook uses the **UCI Bike Sharing Dataset**, hourly version.

The notebook:
- first checks whether `data/hour.csv` already exists,
- if not, it downloads and extracts the dataset,
- then it loads the CSV into a DataFrame.

For a workshop with unreliable internet, place `hour.csv` in:

```text
data/hour.csv
```

before the session begins.

## Environment setup (run once outside class flow)
Install dependencies before class:

```bash
pip install numpy pandas matplotlib scikit-learn ipython
```


In [ ]:
%pip install -q numpy pandas matplotlib scikit-learn ipython


## 0. Setup

In [ ]:
from pathlib import Path
import io
import zipfile
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from sklearn.model_selection import (
    train_test_split,
    KFold,
    cross_validate,
    RandomizedSearchCV,
    TimeSeriesSplit
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


In [ ]:
def load_csv_from_uci_zip(url, suffix, local_path, sep=","):
    """Load a CSV from a local file if present; otherwise download and cache it."""
    local_path = Path(local_path)
    local_path.parent.mkdir(parents=True, exist_ok=True)

    if local_path.exists():
        print(f"Loading cached dataset from: {local_path}")
        return pd.read_csv(local_path, sep=sep)

    print("Local file not found. Downloading dataset...")
    with urllib.request.urlopen(url) as response:
        raw_zip = response.read()

    with zipfile.ZipFile(io.BytesIO(raw_zip)) as zf:
        matching_files = [name for name in zf.namelist() if name.endswith(suffix)]
        if not matching_files:
            raise FileNotFoundError(f"Could not find {suffix} inside the downloaded ZIP file.")

        selected_file = matching_files[0]
        with zf.open(selected_file) as src:
            content = src.read()

    local_path.write_bytes(content)
    print(f"Saved dataset to: {local_path}")
    return pd.read_csv(local_path, sep=sep)


def make_one_hot_encoder(dense=False):
    """Create a OneHotEncoder compatible with older and newer scikit-learn versions."""
    if dense:
        try:
            return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
        except TypeError:
            return OneHotEncoder(handle_unknown="ignore", sparse=False)

    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)


def evaluate_regression(name, y_true, y_pred):
    """Return a one-row DataFrame with common regression metrics."""
    return pd.DataFrame([{
        "Model": name,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
        "R²": r2_score(y_true, y_pred)
    }])


def plot_actual_vs_predicted(y_true, y_pred, title):
    plt.figure(figsize=(7, 5))
    plt.scatter(y_true, y_pred, alpha=0.35)
    lower = min(np.min(y_true), np.min(y_pred))
    upper = max(np.max(y_true), np.max(y_pred))
    plt.plot([lower, upper], [lower, upper], linestyle="--")
    plt.xlabel("Actual rentals")
    plt.ylabel("Predicted rentals")
    plt.title(title)
    plt.show()


def plot_residuals(y_true, y_pred, title):
    residuals = y_true - y_pred
    plt.figure(figsize=(7, 5))
    plt.scatter(y_pred, residuals, alpha=0.35)
    plt.axhline(0, linestyle="--")
    plt.xlabel("Predicted rentals")
    plt.ylabel("Residual = actual - predicted")
    plt.title(title)
    plt.show()


## 1. Load the dataset

The target variable is hourly rental demand.

The data contains:
- calendar information,
- weather conditions,
- normalized continuous weather variables,
- rental counts.

In [ ]:
BIKE_URL = "https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip"

bike = load_csv_from_uci_zip(
    BIKE_URL,
    suffix="hour.csv",
    local_path="data/hour.csv",
    sep=","
)

bike.head()

## 2. Basic dataset inspection

In [ ]:
print("Shape:", bike.shape)
print("\nColumns:")
print(list(bike.columns))
display(bike.head())
display(bike.describe(include="all").T)

### Teaching note
Before modeling, learners should know:
- how many rows and columns exist,
- which column is the target,
- which features are numeric or categorical,
- whether any columns are suspiciously close to the target.

## 3. Check missing values and duplicates

In [ ]:
missing_summary = bike.isna().sum().sort_values(ascending=False)
duplicate_count = bike.duplicated().sum()

print("Missing values per column:")
display(missing_summary)

print(f"Duplicate rows: {duplicate_count}")

The dataset is relatively clean, which is useful pedagogically.  
We will still use imputation inside our preprocessing pipeline because:
1. it is good practice,
2. real datasets are often less tidy,
3. pipelines should be robust to missing values.

## 4. Frame the machine learning problem

In [ ]:
target = "cnt"

print("Target variable:", target)
print("Target description: total hourly bicycle rentals")
display(bike[[target]].describe())

### Why this is a regression problem
The target `cnt` is a **continuous numeric quantity**:
- 0 rentals,
- 25 rentals,
- 180 rentals,
- 500+ rentals.

Therefore, this is a **supervised regression** task.

## 5. Spot target leakage

In [ ]:
bike[["casual", "registered", "cnt"]].head()

The column `cnt` is defined as:

\[
cnt = casual + registered
\]

If we leave `casual` and `registered` in the feature matrix while trying to predict `cnt`, the model would receive the answer indirectly.

That is **data leakage**.

We remove:
- `cnt` because it is the target,
- `casual` and `registered` because they leak the target,
- `instant` because it is only a row identifier.

In [ ]:
drop_columns = ["cnt", "casual", "registered", "instant"]

X = bike.drop(columns=drop_columns)
y = bike[target]

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("Columns retained:")
print(list(X.columns))

## 6. Explore the target distribution

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(y, bins=40)
plt.xlabel("Hourly rentals")
plt.ylabel("Number of observations")
plt.title("Distribution of hourly bike rentals")
plt.show()

print("Target summary:")
display(y.describe())

### Teaching note
The target is right-skewed:
- many hours have low to moderate rentals,
- fewer hours have very high demand.

This matters because:
- large errors on peak-demand hours may dominate RMSE,
- MAE and RMSE can tell slightly different stories.

## 7. Simple exploratory analysis

In [ ]:
eda = bike.copy()
eda["dteday"] = pd.to_datetime(eda["dteday"])

hourly_demand = eda.groupby("hr")["cnt"].mean()
weather_demand = eda.groupby("weathersit")["cnt"].mean()
weekday_demand = eda.groupby("weekday")["cnt"].mean()

display(hourly_demand.rename("average_rentals_by_hour"))
display(weather_demand.rename("average_rentals_by_weather_code"))
display(weekday_demand.rename("average_rentals_by_weekday"))

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(hourly_demand.index, hourly_demand.values, marker="o")
plt.xlabel("Hour of day")
plt.ylabel("Average rentals")
plt.title("Average bike rentals by hour")
plt.xticks(range(0, 24, 2))
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
plt.bar(weather_demand.index.astype(str), weather_demand.values)
plt.xlabel("Weather situation code")
plt.ylabel("Average rentals")
plt.title("Average bike rentals by weather situation")
plt.show()

### Teaching note
This makes the task more tangible:
- demand tends to vary strongly by **hour of day**,
- poor weather reduces demand,
- demand is not simply linear in one variable.

## 8. Feature engineering

In [ ]:
X_model = X.copy()

X_model["dteday"] = pd.to_datetime(X_model["dteday"])
X_model["day_of_month"] = X_model["dteday"].dt.day
X_model["day_of_year"] = X_model["dteday"].dt.dayofyear
X_model = X_model.drop(columns="dteday")

display(X_model.head())
print("Final feature columns:")
print(list(X_model.columns))

### Why engineer date features?
Models cannot directly learn from a raw date string.  
We convert the date into numeric calendar signals:
- day of month,
- day of year.

These help the model learn seasonality and gradual changes over the year.

## 9. Define feature groups

In [ ]:
categorical_features = [
    "season", "yr", "mnth", "hr",
    "holiday", "weekday", "workingday", "weathersit"
]

numeric_features = [
    col for col in X_model.columns
    if col not in categorical_features
]

print("Categorical features:")
print(categorical_features)

print("\nNumeric features:")
print(numeric_features)

### Important modeling detail
Several columns are stored as integers but should be treated as **categories**:
- hour,
- weekday,
- month,
- weather code,
- working-day flag.

Treating them as numeric in a linear model would imply artificial ordering and distances.

## 10. Build preprocessing pipelines

In [ ]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline_sparse = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", make_one_hot_encoder(dense=False))
])

categorical_pipeline_dense = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", make_one_hot_encoder(dense=True))
])

preprocessor_sparse = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline_sparse, categorical_features)
])

preprocessor_dense = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline_dense, categorical_features)
])

preprocessor_sparse

### Why two preprocessors?
- Sparse one-hot output is efficient for linear models and random forests.
- Dense output is used for `HistGradientBoostingRegressor`, which expects dense arrays after preprocessing in this setup.

## 11. Split into training and test sets

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_model,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE
)

print("Training features:", X_train.shape)
print("Test features:", X_test.shape)
print("Training target:", y_train.shape)
print("Test target:", y_test.shape)

### Teaching note
The test set is kept untouched until final evaluation.  
Model selection and hyperparameter tuning happen using only the training data.

## 12. Baseline model

In [ ]:
baseline_model = Pipeline(steps=[
    ("preprocessor", preprocessor_sparse),
    ("model", DummyRegressor(strategy="mean"))
])

baseline_model.fit(X_train, y_train)
baseline_pred = baseline_model.predict(X_test)

baseline_results = evaluate_regression(
    "Mean baseline",
    y_test,
    baseline_pred
)

baseline_results

### Why the baseline matters
A baseline answers:

> Is our ML model better than a trivial default?

Here, the baseline always predicts the **average training-set demand**.

## 13. Linear regression

In [ ]:
linear_model = Pipeline(steps=[
    ("preprocessor", preprocessor_sparse),
    ("model", LinearRegression())
])

linear_model.fit(X_train, y_train)
linear_pred = linear_model.predict(X_test)

linear_results = evaluate_regression(
    "Linear regression",
    y_test,
    linear_pred
)

linear_results

### Why linear regression?
It is:
- simple,
- interpretable,
- a useful first real model.

But bike rental demand is likely non-linear:
- rush-hour demand,
- weather interactions,
- seasonality.

## 14. Random forest regression

In [ ]:
forest_model = Pipeline(steps=[
    ("preprocessor", preprocessor_sparse),
    ("model", RandomForestRegressor(
        n_estimators=180,
        max_depth=None,
        min_samples_leaf=1,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

forest_model.fit(X_train, y_train)
forest_pred = forest_model.predict(X_test)

forest_results = evaluate_regression(
    "Random forest",
    y_test,
    forest_pred
)

forest_results

### Why a random forest?
Random forests can capture:
- non-linear patterns,
- feature interactions,
- complex split-based decision rules.

They are often much stronger than a linear model on structured/tabular data.

## 15. More sophisticated model: Histogram Gradient Boosting

In [ ]:
hist_gb_model = Pipeline(steps=[
    ("preprocessor", preprocessor_dense),
    ("model", HistGradientBoostingRegressor(
        learning_rate=0.08,
        max_iter=250,
        max_leaf_nodes=31,
        min_samples_leaf=20,
        l2_regularization=0.1,
        early_stopping=True,
        random_state=RANDOM_STATE
    ))
])

hist_gb_model.fit(X_train, y_train)
hist_gb_pred = hist_gb_model.predict(X_test)

hist_gb_results = evaluate_regression(
    "HistGradientBoosting",
    y_test,
    hist_gb_pred
)

hist_gb_results

### Why this is a more sophisticated model
Histogram Gradient Boosting builds an additive ensemble of trees:
- each new tree focuses on reducing errors left by previous trees,
- it can model strong non-linear structure,
- it supports regularization and early stopping,
- it is often a strong choice for tabular regression.

This makes it a useful step beyond random forests.

## 16. Compare all models

In [ ]:
comparison = pd.concat(
    [
        baseline_results,
        linear_results,
        forest_results,
        hist_gb_results
    ],
    ignore_index=True
).sort_values("RMSE")

comparison

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(comparison["Model"], comparison["RMSE"])
plt.ylabel("RMSE")
plt.title("Model comparison by RMSE")
plt.xticks(rotation=20)
plt.show()

### Interpreting the metrics
- **MAE**: average absolute error, easy to explain.
- **RMSE**: penalizes larger errors more strongly.
- **R²**: proportion of variance explained compared with a mean-only model.

For operational forecasting, RMSE is useful when large misses are especially costly.

## 17. Diagnose predictions visually

In [ ]:
plot_actual_vs_predicted(
    y_test,
    hist_gb_pred,
    "Histogram Gradient Boosting: Actual vs Predicted"
)

In [ ]:
plot_residuals(
    y_test,
    hist_gb_pred,
    "Histogram Gradient Boosting: Residual Plot"
)

### What to look for
- If predictions cluster near the diagonal, the model is doing well.
- Systematic curves or fan shapes in residuals suggest remaining structure or heteroscedasticity.
- Large residuals during peak-demand periods may deserve closer inspection.

## 18. Cross-validation for model stability

In [ ]:
cv = KFold(
    n_splits=3,
    shuffle=True,
    random_state=RANDOM_STATE
)

cv_scores_hist = cross_validate(
    hist_gb_model,
    X_train,
    y_train,
    cv=cv,
    scoring={
        "mae": "neg_mean_absolute_error",
        "rmse": "neg_root_mean_squared_error",
        "r2": "r2"
    },
    n_jobs=-1,
    return_train_score=False
)

cv_summary = pd.DataFrame({
    "Fold MAE": -cv_scores_hist["test_mae"],
    "Fold RMSE": -cv_scores_hist["test_rmse"],
    "Fold R²": cv_scores_hist["test_r2"]
})

display(cv_summary)
display(cv_summary.agg(["mean", "std"]))

### Why cross-validation?
A single train/test split can be lucky or unlucky.  
Cross-validation gives a more stable picture of expected model performance.

## 19. Hyperparameter tuning for the advanced model

In [ ]:
param_distributions = {
    "model__learning_rate": [0.03, 0.05, 0.08, 0.10],
    "model__max_iter": [150, 250, 350],
    "model__max_leaf_nodes": [15, 31, 63],
    "model__min_samples_leaf": [10, 20, 35],
    "model__l2_regularization": [0.0, 0.05, 0.10, 0.25],
    "model__max_depth": [None, 6, 10]
}

tuning_model = Pipeline(steps=[
    ("preprocessor", preprocessor_dense),
    ("model", HistGradientBoostingRegressor(
        early_stopping=True,
        random_state=RANDOM_STATE
    ))
])

search = RandomizedSearchCV(
    estimator=tuning_model,
    param_distributions=param_distributions,
    n_iter=12,
    scoring="neg_root_mean_squared_error",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
    refit=True
)

search.fit(X_train, y_train)

print("Best CV RMSE:", -search.best_score_)
print("Best parameters:")
display(search.best_params_)

### Hyperparameter concepts being demonstrated
- `learning_rate`: how much each boosting stage contributes.
- `max_iter`: number of boosting iterations.
- `max_leaf_nodes`: tree complexity.
- `min_samples_leaf`: regularization through minimum leaf size.
- `l2_regularization`: penalty to reduce overfitting.
- `max_depth`: explicit tree depth control.

`RandomizedSearchCV` samples combinations rather than exhaustively trying every possibility.

## 20. Evaluate the tuned advanced model

In [ ]:
best_model = search.best_estimator_
tuned_pred = best_model.predict(X_test)

tuned_results = evaluate_regression(
    "Tuned HistGradientBoosting",
    y_test,
    tuned_pred
)

final_comparison = pd.concat(
    [
        comparison,
        tuned_results
    ],
    ignore_index=True
).sort_values("RMSE")

final_comparison

In [ ]:
plot_actual_vs_predicted(
    y_test,
    tuned_pred,
    "Tuned Histogram Gradient Boosting: Actual vs Predicted"
)

In [ ]:
plot_residuals(
    y_test,
    tuned_pred,
    "Tuned Histogram Gradient Boosting: Residual Plot"
)

### Teaching point
Tuning does **not** always produce a dramatic gain.  
That itself is an important lesson:
- sometimes the default model is already good,
- sometimes the search space is weak,
- sometimes the dataset limits performance,
- sometimes feature engineering matters more than tuning.

## 21. Inspect model sensitivity with permutation importance

In [ ]:
perm = permutation_importance(
    best_model,
    X_test,
    y_test,
    n_repeats=5,
    random_state=RANDOM_STATE,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

importance = pd.DataFrame({
    "feature": X_test.columns,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std
}).sort_values("importance_mean", ascending=False)

importance

In [ ]:
top_importance = importance.head(12).sort_values("importance_mean")

plt.figure(figsize=(8, 6))
plt.barh(top_importance["feature"], top_importance["importance_mean"])
plt.xlabel("Increase in negative RMSE score when shuffled")
plt.title("Permutation importance — tuned boosted model")
plt.show()

### How to explain permutation importance
For each feature:
1. shuffle that feature in the test set,
2. measure how much performance worsens,
3. larger degradation means the model relied more heavily on that feature.

This is useful for more opaque non-linear models.

## 22. Error analysis by hour of day

In [ ]:
error_analysis = X_test.copy()
error_analysis["actual"] = y_test.values
error_analysis["predicted"] = tuned_pred
error_analysis["absolute_error"] = np.abs(error_analysis["actual"] - error_analysis["predicted"])

hour_error = (
    error_analysis
    .groupby("hr")["absolute_error"]
    .mean()
    .sort_index()
)

display(hour_error.rename("mean_absolute_error_by_hour"))

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(hour_error.index, hour_error.values, marker="o")
plt.xlabel("Hour of day")
plt.ylabel("Mean absolute error")
plt.title("Where does the tuned model make larger errors?")
plt.xticks(range(0, 24, 2))
plt.show()

### Teaching note
A global metric can hide local weaknesses.  
Error analysis asks:
- does the model struggle during rush hour?
- does it struggle more at night?
- does it underperform in rare conditions?

## 23. Optional methodological note: random split vs time-aware split

This notebook uses a **random split** because it is simpler for an introductory ML session.

However, bike demand is time-indexed.  
For a real forecasting problem, a chronological split may be more appropriate:

- train on earlier dates,
- test on later dates.

That setup more closely resembles deployment and avoids learning from the future.

In [ ]:
# Optional illustration only — not used in the main workflow
time_ordered = bike.copy()
time_ordered["dteday"] = pd.to_datetime(time_ordered["dteday"])
time_ordered = time_ordered.sort_values(["dteday", "hr"])

time_ordered[["dteday", "hr", "cnt"]].head()

### Optional code demo: TimeSeriesSplit fold windows
This shows how chronological folds differ from random KFold.


In [ ]:
time_ordered_cv = bike.copy()
time_ordered_cv["dteday"] = pd.to_datetime(time_ordered_cv["dteday"])
time_ordered_cv = time_ordered_cv.sort_values(["dteday", "hr"]).reset_index(drop=True)

tscv = TimeSeriesSplit(n_splits=3)
for fold, (train_idx, val_idx) in enumerate(tscv.split(time_ordered_cv), start=1):
    train_start = time_ordered_cv.loc[train_idx[0], "dteday"].date()
    train_end = time_ordered_cv.loc[train_idx[-1], "dteday"].date()
    val_start = time_ordered_cv.loc[val_idx[0], "dteday"].date()
    val_end = time_ordered_cv.loc[val_idx[-1], "dteday"].date()
    print(
        f"Fold {fold}: train {train_start} -> {train_end} | "
        f"validate {val_start} -> {val_end}"
    )


## 24. Summary

By the end of this notebook, we have:
- framed bike demand prediction as a regression task,
- removed leakage columns,
- engineered features,
- built reusable preprocessing pipelines,
- compared baseline, linear, random forest, and boosted models,
- tuned a sophisticated model using randomized cross-validation,
- evaluated with MAE, RMSE, and R²,
- interpreted model behavior using permutation importance,
- performed targeted error analysis.

This is a complete end-to-end machine learning workflow for tabular regression.

## 25. Suggested live-teaching pauses

Use these pause points during delivery:

1. **After leakage detection**  
   Ask: “What would happen if we kept `casual` and `registered`?”

2. **After linear regression results**  
   Ask: “Why might a linear model struggle here?”

3. **After random forest vs boosting**  
   Ask: “What kinds of patterns can trees capture that a line cannot?”

4. **After tuning**  
   Ask: “Did tuning matter more than model choice?”

5. **After permutation importance**  
   Ask: “Do the important features make domain sense?”